## 基于提示词的结构化输出

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("DASHSCOPE_API_KEY")

In [ ]:
from builtins import print

from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_community.chat_models import ChatTongyi

system_prompt = """
# 身份

- 你是一个科幻作家，根据用户的要求创建一个天空之都

# 指令
- 请务必以JSON格式输出，不要加任何markdown样式。

# 示例：
user： 月球的首都是什么？
assistant：
{
    "name"："月华市（Lunaria）",
    "location": "位于月球正面附近的静海基地遗址之上，依托巨大的穹顶与地下网络建成",
    "vibe": "冷冽、高效、革新",
    "economy": "氦-3能源开采、量子通信枢纽、尖端生物圈农业"
}

"""

# 1. 初始化千问大模型
model = ChatTongyi(model="qwen-plus")

# 2. 创建Agent时，需要传入实例化的模型对象，而不是字符串名称
agent = create_agent(
    model=model,
    system_prompt=system_prompt
)


response = agent.invoke(
    {
        "messages": [HumanMessage(content="金星的首都是什么？")]
    }
)

print(response)


In [ ]:
print(response['messages'][-1].content)

## 基于model的结构化输出

In [ ]:
from pydantic import BaseModel


# 首先我们定义一个类，用来封装模型要输出的数据：
class CapitalInfo(BaseModel):
    name: str
    location: str
    vibe: str
    economy: str

In [ ]:
# 我们可以创建智能体时设置结构化输出的格式，LangChain会自动帮我们完成提示词改造和响应结果解析。

model = ChatTongyi(model="qwen-plus")

# 直接使用 LLM 对象调用即可
messages = [
    {"role": "system", "content": system_prompt},
    HumanMessage(content="金星的首都是什么？")
]

response = model.invoke(messages)

print(response.content)

# agent = create_agent(
#     model = model,
#     system_prompt = "你是一个科幻作家，根据用户的要求创建一个太空之都。",
#     response_format = CapitalInfo  # 设置结构化输出的格式
# )
#
# response = agent.invoke(
#     {"messages": [HumanMessage(content="月球的首都是什么？")]}
# )
#
# # 输出结果
# print(response)

## 结合工具搜索与结构化输出

In [16]:
import os
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_community.tools.tavily_search import TavilySearchResults
from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage
from langchain_community.chat_models import ChatTongyi

# 确保你已经在环境变量中配置了 TAVILY_API_KEY
# os.environ["TAVILY_API_KEY"] = "your-tavily-api-key"

## 自定义tavily工具
tavily = TavilySearchResults(
    max_results=5,
    topic="general",
)

# 自定封装tool
@tool
def web_search(query: str) -> str:
    """Search the web for information"""
    return tavily.invoke(query)

# Agent回答内容引用的网页信息
class Reference(BaseModel):
    title: str = Field(description="The title of the web page cited in the answer")
    url: str = Field(description="The url of the web page cited in the answer")

# Agent的回答内容
class AnswerInfo(BaseModel):
    answer: str = Field(description="The final answer for user")
    reference: list[Reference] = Field(description="The web pages cited in the answer")

# 创建智能体，使用预定义的tavily工具
model = ChatTongyi(model="qwen-plus")

# 【核心修改】：因为阿里通义大模型 API 当前不支持同时绑定 tools 和强制的 response_format，
# 我们采用两步走策略：
# 第 1 步：让带有工具的 Agent 去搜索并给出自由格式的回答。
agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt="你是一个智能助手，必须使用 web_search 工具搜索信息，并详细解答用户的问题，并在回答中附上参考链接。"
)

print("1. 正在思考与搜索中...")
agent_response = agent.invoke({
    "messages": [HumanMessage(content="蒸蚌是什么梗？")]
})
raw_answer = agent_response["messages"][-1].content
print("搜索完成！提取原始答案...")

# 第 2 步：将自由格式的回答交给另一个带有结构化输出能力（with_structured_output）的模型进行解析提炼
print("2. 正在将其转换为结构化数据...")
structured_model = model.with_structured_output(AnswerInfo)
final_result = structured_model.invoke(
    f"请从以下回答中提取出用户的答案，并提取出引用的网页信息：\n{raw_answer}"
)

print("\n--- 最终结构化结果 ---")
print(final_result)
# 验证并创建对象
# parsed_data = AnswerInfo(**final_result)
# print(parsed_data.answer)  # 可以正常输出文本
# print(parsed_data.reference[0].title)  # 可以提取第一个引用的标题

1. 正在思考与搜索中...


ValueError: request_id: 70a25721-7b53-9fd8-b171-b203cdb22032 
 status_code: 400 
 code: InvalidParameter 
 message: url error, please check url！ For details, see: https://help.aliyun.com/zh/model-studio/error-code#error-url